In [62]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from sklearn.datasets import make_classification
import torch

In [63]:
torch.manual_seed(42)

In [64]:
df = pd.read_csv("/content/fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [65]:
# Google Drive mounting is not needed since '/content/fmnist_small.csv' is already local.

In [66]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [68]:
X_train = X_train/255.0
X_test = X_test/255.0

In [69]:
from torch.utils.data import Dataset, DataLoader

In [70]:
#create a CustomDataset class
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [71]:
#create train_dataset object
train_dataset = CustomDataset(X_train,y_train)

In [72]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)


In [73]:
#create train and test loader can be outside for first half optuna
train_loader = DataLoader(train_dataset, batch_size = 32 ,shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle =False)

In [74]:
class MyNN1(nn.Module):

  def __init__(self,input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate=0.3):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate)) # Using the dynamic dropout rate
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer,output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self,x):
    return self.model(x)

In [75]:
# #objective function

# def objective(trial):

#   #extract hyperparameter from the search space
#   num_hidden_layers = trial.suggest_int("num_hidden_layer",1,5)
#   neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)

#   #model initialize
#   input_dim = 784
#   output_dim = 10

#   # Fixed class spelling (MyNN1 instead of myNN1)
#   model = MyNN1(input_dim,output_dim,num_hidden_layers,neurons_per_layer)

#   #parameter initialize
#   learning_rate = 0.01
#   epochs = 50

#   #optimizer selection
#   loss_function = nn.CrossEntropyLoss()
#   # Fixed optimizer to use the local model instead of global mynn
#   optimizer = optim.SGD(model.parameters(), lr=learning_rate,weight_decay = 1e-4)

#   #training loop
#   for epoch in range(epochs):
#     # Indented the training step inside the epoch loop
#     for batch_train , batch_label in train_loader:
#       #forward pass
#       y_pred = model(batch_train)

#       #calculate loss
#       loss = loss_function(y_pred ,batch_label)

#       #back pass
#       optimizer.zero_grad()
#       loss.backward()

#       #upgrade grades
#       optimizer.step()

#   #evaluation
#   model.eval()

#   total = 0
#   correct = 0

#   with torch.no_grad():
#     for batch_features, batch_labels in test_loader:
#       # Fixed evaluation to use local model instead of global mynn
#       outputs = model(batch_features)

#       _, predicted = torch.max(outputs, 1)

#       total = total + batch_labels.shape[0]

#       correct = correct + (predicted == batch_labels).sum().item()

#   accuracy = (correct/total)

#   return accuracy

In [76]:
!pip install optuna


In [77]:
# import optuna

# study = optuna.create_study(direction = 'maximize')
# study.optimize(objective,n_trials=10)

In [78]:
# study.best_value

In [79]:
# study.best_params

In [80]:
#objective function

def objective(trial):

  #extract hyperparameter from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layer",1,5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)
  epochs = trial.suggest_int("epochs",10,50,step=10)
  # Fixed: Changed suggest_int to suggest_float to allow decimal learning rates
  learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1,log=True)
  dropout_rate = trial.suggest_float("dropout_rate",0.1,0.5, step = 0.1)
  batch_size = trial.suggest_categorical("batch_size",[16,32,64,128])
  optimizer_name = trial.suggest_categorical("optimizer_name",['Adam','SGD','RMSprop'])
  weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3)


  train_loader = DataLoader(train_dataset, batch_size = batch_size ,shuffle = True)
  test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle =False) # Changed train_dataset to test_dataset

  #model initialize
  input_dim = 784
  output_dim = 10

  # Fixed class spelling (MyNN1 instead of myNN1)
  model = MyNN1(input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate)

  #parameter initialize


  #optimizer selection
  loss_function = nn.CrossEntropyLoss()

  # Fixed: Assigned the selected optimizer to the variable 'optimizer'
  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate,weight_decay = weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate,weight_decay = weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate,weight_decay = weight_decay)

  #training loop
  for epoch in range(epochs):
    # Indented the training step inside the epoch loop
    for batch_train , batch_label in train_loader:
      #forward pass
      y_pred = model(batch_train)

      #calculate loss
      loss = loss_function(y_pred ,batch_label)

      #back pass
      optimizer.zero_grad()
      loss.backward()

      #upgrade grades
      optimizer.step()

  #evaluation
  model.eval()

  total = 0
  correct = 0

  with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      # Fixed evaluation to use local model instead of global mynn
      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

  accuracy = (correct/total)

  return accuracy

In [82]:
import optuna
import torch.optim as optim

study = optuna.create_study(direction = 'maximize')
study.optimize(objective,n_trials=10)

[I 2026-10-08 14:19:51,254] A new study created in memory with name: no-name-ae68169a-bcc1-479f-b5f1-a4c598f792d5
[I 2026-10-08 14:20:33,816] Trial 0 finished with value: 0.8383333333333334 and parameters: {'num_hidden_layer': 3, 'neurons_per_layer': 104, 'epochs': 40, 'learning_rate': 0.0003807784006299614, 'dropout_rate': 0.5, 'batch_size': 16, 'optimizer_name': 'RMSprop', 'weight_decay': 0.00044791624493014845}. Best is trial 0 with value: 0.8383333333333334.
[I 2026-10-08 14:20:39,787] Trial 1 finished with value: 0.2966666666666667 and parameters: {'num_hidden_layer': 2, 'neurons_per_layer': 104, 'epochs': 10, 'learning_rate': 1.2141914152273403e-05, 'dropout_rate': 0.5, 'batch_size': 16, 'optimizer_name': 'SGD', 'weight_decay': 0.00029998388021456533}. Best is trial 0 with value: 0.8383333333333334.
[I 2026-10-08 14:20:44,298] Trial 2 finished with value: 0.8025 and parameters: {'num_hidden_layer': 3, 'neurons_per_layer': 112, 'epochs': 20, 'learning_rate': 0.0010517079815056253,

In [ ]:
from torch.nn.modules.batchnorm import BatchNorm1d
from torch.nn.modules.activation import ReLU
#define NN class
class MyNN(nn.Module):
  def __init__(self,num_features,dropout_rate):

    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
        nn.Linear(64,10),

    )
    ##we dont need to explicitly define softmax here as in pytorch in cross entropy loss has softmax implemented internally
  def forward(self,X):
    return self.model(X)

In [ ]:
# set learning rate and epochs
learning_rate = 0.01
epochs = 100

In [83]:
# Retrieve the best hyperparameters from the Optuna study
best_params = study.best_params
print("Best Parameters Found:", best_params)

# Instantiate the optimized model using the best hyperparameters found
mynn = MyNN1(
    input_dim=784,
    output_dim=10,
    num_hidden_layers=best_params['num_hidden_layer'],
    neurons_per_layer=best_params['neurons_per_layer'],
    dropout_rate=best_params['dropout_rate']
)

# Loss function
loss_function = nn.CrossEntropyLoss()

# Set learning rate and epochs based on hyperparameter tuning
learning_rate = best_params['learning_rate']
epochs = best_params['epochs']
batch_size = best_params['batch_size']

# Dynamic Dataloader setup with best batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Optimizer selection based on the best trial
optimizer_name = best_params['optimizer_name']
weight_decay = best_params['weight_decay']

if optimizer_name == 'Adam':
    optimizer = optim.Adam(mynn.parameters(), lr=learning_rate, weight_decay=weight_decay)
elif optimizer_name == 'SGD':
    optimizer = optim.SGD(mynn.parameters(), lr=learning_rate, weight_decay=weight_decay)
else:
    optimizer = optim.RMSprop(mynn.parameters(), lr=learning_rate, weight_decay=weight_decay)

Best Parameters Found: {'num_hidden_layer': 5, 'neurons_per_layer': 88, 'epochs': 50, 'learning_rate': 0.0013466234680809311, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer_name': 'Adam', 'weight_decay': 0.00023283834619201337}


In [84]:
# Training loop with optimized parameters
print(f"Training model for {epochs} epochs...")
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_train, batch_label in train_loader:
        # Forward pass
        y_pred = mynn(batch_train)

        # Calculate loss
        loss = loss_function(y_pred, batch_label)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Upgrade weights
        optimizer.step()
        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}')

Training model for 50 epochs...
Epoch 10/50 - Loss: 0.6345
Epoch 20/50 - Loss: 0.4877
Epoch 30/50 - Loss: 0.4576
Epoch 40/50 - Loss: 0.4270
Epoch 50/50 - Loss: 0.3788


In [ ]:
#set model to eval mode
mynn.eval()
#there are certain features which are different in eval and train, dropout, mean-std calculation

In [85]:
import torch

# Evaluation code on test dataset
mynn.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    outputs = mynn(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

test_accuracy = correct / total
print(f"Final Accuracy on Test Data: {test_accuracy:.4f} ({correct}/{total})")

Final Accuracy on Test Data: 0.8275 (993/1200)


In [86]:
# Evaluation code on train dataset
mynn.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    outputs = mynn(batch_features)
    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

train_accuracy = correct / total
print(f"Final Accuracy on Train Data: {train_accuracy:.4f} ({correct}/{total})")

Final Accuracy on Train Data: 0.9419 (4521/4800)
